In [6]:
from textwrap import dedent
from agno.agent import Agent
from agno.models.groq import Groq
from agno.tools.duckduckgo import DuckDuckGoTools
from agno.tools.newspaper4k import Newspaper4kTools
from datetime import datetime
import openai
import requests
import re

In [9]:
import os
from dotenv import load_dotenv

load_dotenv()

# Retrieve API keys
groq_api_key = os.getenv('GROQ_API_KEY')
phi_api_key = os.getenv('PHI_API_KEY')
openai_api_key = os.getenv('OPENAI_API_KEY')

print("API keys have been set!")

API keys have been set!


In [10]:

# Check if the API keys are set
print("Groq API key set:" if os.getenv("GROQ_API_KEY") else "Groq API key not set")
print("PHI API key set:" if os.getenv("PHI_API_KEY") else "PHI API key not set")
print("OpenAI API key set:" if os.getenv("OPENAI_API_KEY") else "OpenAI API key not set")


Groq API key set:
PHI API key set:
OpenAI API key set:


### Research Agent (Web Search & Newspaper Search)

In [ ]:
# user query
user_query = "Analyze the current updates on Chittagong Port including operational trends, policy changes, and logistical disruptions."

# Function to extract port details from the query using regex.
def extract_port(query: str) -> str:
    # The regex pattern here is simplistic and assumes that port names follow patterns like "Port of ..."
    # You can adjust this pattern for more robust matching or use an NLP library for improved accuracy.
    pattern = r"(Port of [A-Za-z\s&-]+)"
    match = re.search(pattern, query)
    if match:
        return match.group(1).strip()
    return "Port of New York and New Jersey"  # Default port if none detected

# Extract the target port from the query
target_port = extract_port(user_query)

# Define the dynamic research agent for supply chain operations at any port
research_agent = Agent(
    model=Groq(id="llama3-70b-8192"),
    tools=[DuckDuckGoTools(), Newspaper4kTools()],
    description=dedent(f"""\
        You are a Supply Chain Analyst specialized in monitoring real-time operational updates and trends 
        at high-traffic ports globally. Your current focus is on analyzing operations at {target_port}.
        Your expertise includes:

        - Deep investigative research on supply chain operations,
        - Fact-checking and source verification,
        - Data-driven reporting on current logistics trends,
        - Expert analysis of port operations impacting supply chains,
        - Trend analysis and implications for near-future operations,
        - Simplification of complex logistics and operational concepts,
        - Ensuring balanced and ethical perspectives,
        - Integrating global context to supply chain dynamics.
    """),
    instructions=dedent(f"""\
        1. Research Phase
           - Conduct web searches and gather the latest news and updates on port operations globally, then narrow down to {target_port}.
           - Prioritize real-time, authoritative, and non-historic sources.
           - Focus on current logistical challenges, operational disruptions, and policy changes that directly impact supply chain efficiency at {target_port}.

        2. Analysis Phase
           - Extract critical data points regarding current operations, policy adjustments, and logistical updates for {target_port}.
           - Analyze trends in real-time supply chain disruptions or operational improvements.
           - Assess the immediate impact of geopolitical factors and market conditions on port throughput and overall logistics flow.

        3. Writing Phase
           - Develop a compelling headline that reflects the latest operational updates at {target_port}.
           - Structure the report with clear sections focused on current operations:
             - Executive Summary, Critical Updates, Impact Analysis, and Future Outlook.
           - Include relevant quotes, statistics, and near-term implications.

        4. Quality Control
           - Verify all facts with cross-reference from multiple reputable sources.
           - Emphasize current operational details without including extensive historical or unrelated background.
           - Provide clear context for all data and offer actionable insights on operational trends.
    """),
    expected_output=dedent(f"""\
        # {{Compelling Headline on Current Port Operations and Supply Chain Impact}}

        ## Executive Summary
        {{Concise overview of the latest findings regarding operations at {target_port}.}}

        ## Critical Updates
        {{Recent operational changes and logistical disruptions at {target_port} that directly impact supply chains.}}
        {{Breaking news on policy changes and immediate adjustments by port authorities.}}
        {{Statistical evidence on port throughput and current operational trends.}}

        ## Impact Analysis
        {{Immediate implications on supply chain operations and logistics.}}
        {{Stakeholder perspectives including port authorities and key industry players.}}
        {{Analysis of market, geopolitical, and economic influences.}}

        ## Future Outlook
        {{Emerging trends and upcoming changes in port operations at {target_port}.}}
        {{Expert predictions on near-future performance and policy modifications.}}
        {{Potential challenges and actionable opportunities for supply chain optimization.}}

        ## Expert Insights
        {{Notable quotes and analysis from port operation experts and supply chain analysts.}}
        {{Discussion on contrasting viewpoints regarding current operational shifts.}}

        ## Sources & Methodology
        {{List of primary sources with key contributions.}}
        {{Overview of research methodology and data verification for current updates.}}
        
        ---
        Research conducted by Supply Chain Analyst  
        Logistics Analysis Report  
        Published: {{current_date}}  
        Last Updated: {{current_time}}
    """),
    markdown=True,
    show_tool_calls=True,
    add_datetime_to_instructions=True,
)

if __name__ == "__main__":
    # Use the user's query to analyze the updates for the detected port details
    research_agent.print_response(user_query, stream=True)

Output()

In [ ]:
from typing import Optional
from llama_index import ServiceContext, VectorStoreIndex
from llama_index.agent import OpenAIAgent  # Using OpenAIAgent instead of ReActAgent
from llama_index.llms import Groq
from llama_index.llms import OpenAI, ChatMessage
from llama_index.tools import BaseTool, FunctionTool
import nest_asyncio
from llama_index.core import Settings
from llama_index.readers import SimpleDirectoryReader
from llama_index import download_loader
import regex as re
from datetime import datetime
import logging
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Retrieve API keys
groq_api_key = os.getenv('GROQ_API_KEY')
phi_api_key = os.getenv('PHI_API_KEY')

# Current UTC time and user
CURRENT_UTC = "2025-02-20 02:42:58"
CURRENT_USER = "pritom02bh"

# Download necessary loaders
NewsReader = download_loader("NewsReader")
DuckDuckGoReader = download_loader("DuckDuckGoReader")

class PortAnalysisAgent:
    def __init__(self):
        # Initialize Groq LLM
        self.llm = Groq(
            api_key=groq_api_key,
            model_name="llama3-70b-8192"
        )
        
        # Create service context
        self.service_context = ServiceContext.from_defaults(
            llm=self.llm,
            system_prompt=self.get_system_prompt()
        )
        
        # Initialize tools
        self.news_reader = NewsReader()
        self.search_reader = DuckDuckGoReader()
        
    def get_system_prompt(self):
        """Generate system prompt with current time and user."""
        return f"""
        You are a Supply Chain Analyst specialized in monitoring real-time operational updates 
        and trends at ports globally. 
        
        Current UTC Time: {CURRENT_UTC}
        Current User: {CURRENT_USER}
        
        Your expertise includes:
        - Deep investigative research on supply chain operations
        - Fact-checking and source verification
        - Data-driven reporting on current logistics trends
        - Expert analysis of port operations impacting supply chains
        - Trend analysis and implications for near-future operations
        - Simplification of complex logistics and operational concepts
        - Ensuring balanced and ethical perspectives
        - Integrating global context to supply chain dynamics
        """
    
    def extract_port(self, query: str) -> str:
        """Extract port name from query using regex."""
        pattern = r"(Port of [A-Za-z\s&-]+)"
        match = re.search(pattern, query)
        return match.group(1).strip() if match else "Port of Chittagong"

    def create_tools(self, target_port: str):
        """Create custom tools for port analysis."""
        
        def search_port_news():
            """Search for latest news about the target port."""
            search_results = self.search_reader.load_data(
                f"latest news {target_port} operations logistics"
            )
            return search_results

        def analyze_port_operations():
            """Analyze current port operations and trends."""
            current_news = self.news_reader.load_data(
                urls=[r.source for r in search_port_news()][:3]
            )
            return current_news

        return [
            FunctionTool.from_defaults(
                fn=search_port_news,
                name="search_port_news",
                description="Searches for latest news and updates about the port"
            ),
            FunctionTool.from_defaults(
                fn=analyze_port_operations,
                name="analyze_port_operations",
                description="Analyzes current port operations and trends"
            )
        ]

    def create_agent(self, target_port: str):
        """Create a LlamaIndex agent for port analysis."""
        
        tools = self.create_tools(target_port)
        
        # Create agent with tools
        agent = OpenAIAgent.from_tools(
            tools=tools,
            llm=self.llm,
            verbose=True,
            system_prompt=self.get_system_prompt()
        )
        
        return agent

    def analyze_port(self, query: str) -> str:
        """Main method to analyze port operations based on user query."""
        
        # Extract target port
        target_port = self.extract_port(query)
        logger.info(f"Analyzing updates for: {target_port}")
        
        # Create and run agent
        agent = self.create_agent(target_port)
        
        # Format the specific query for the agent
        formatted_query = f"""
        Analyze the current updates on {target_port} including:
        1. Latest operational trends
        2. Recent policy changes
        3. Current logistical disruptions
        4. Throughput metrics
        5. Supply chain impacts
        
        Format the response in markdown with the following sections:
        - Executive Summary
        - Critical Updates
        - Impact Analysis
        - Future Outlook
        - Expert Insights
        - Sources & Methodology
        """
        
        response = agent.chat(formatted_query)
        return response.response

def main():
    # Initialize the agent system
    port_analyzer = PortAnalysisAgent()
    
    # Example query
    user_query = "Analyze the current updates on Chittagong Port including operational trends, policy changes, and logistical disruptions."
    
    # Get analysis
    result = port_analyzer.analyze_port(user_query)
    print(result)

if __name__ == "__main__":
    main()

ImportError: cannot import name 'ReActAgent' from 'llama_index.agent' (unknown location)

In [16]:
! pip install llama-index-tools

ERROR: Could not find a version that satisfies the requirement llama-index-tools (from versions: none)
ERROR: No matching distribution found for llama-index-tools
